In [1]:
import pandas
import Dependency.Data_splitting as spl
from Dependency.one_hot_encoding import one_hot_encoding
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import GridSearchCV,HalvingGridSearchCV
from sklearn.decomposition import PCA

dataset = spl.selecting()

encoded = one_hot_encoding(dataset["training_data"], dataset["testing_data"])
dataset["training_data"] = encoded["training"]
dataset["testing_data"] = encoded["testing"]

dataset = spl.targeting(dataset)

PCA_execute = True

NaiveBayes = not PCA_execute
Forest = True
HistGB = False
SVM = False

if(PCA_execute):
    #normalization of data for pca
    dataset["training_data_X"] = spl.normalization(dataset["training_data_X"])
    dataset["testing_data_X"] = spl.normalization(dataset["testing_data_X"])

    #PCA
    pca = PCA(n_components=0.99)
    dataset["training_data_X"] = pandas.DataFrame(pca.fit_transform(dataset["training_data_X"]))
    dataset["testing_data_X"] = pandas.DataFrame(pca.transform(dataset["testing_data_X"]))

data_training_X = dataset["training_data_X"]
data_training_Y =dataset["training_data_Y"]

pandas.set_option('display.max_colwidth', None)

cols = ["params", "mean_test_score", "std_test_score", "rank_test_score"]
results = pandas.DataFrame(columns=cols)

In [2]:
#Categorical_Naive_Bayes Grid

from sklearn.naive_bayes import CategoricalNB

p_grid = {
    "alpha":[0.5,1.0,2.0,5.0,10.0,15.0,20.0,25.0,30.0,40.0],
}

C_NB_grid = GridSearchCV(
    estimator=CategoricalNB(),
    param_grid=p_grid,
    scoring='accuracy',
    n_jobs=2)

def coord_positive(dataset):
    for column in dataset.columns.to_list():
        if(dataset[column].min()<0):
            c = dataset[column]
            dataset[column] = c-dataset[column].min()
    return dataset

data_training_positive = coord_positive(data_training_X)

if(NaiveBayes):
    C_NB_grid.fit(data_training_positive, data_training_Y)
    results = pandas.DataFrame(C_NB_grid.cv_results_)
    
results[cols].sort_values("rank_test_score").head(10)

,params,mean_test_score,std_test_score,rank_test_score


In [ ]:
#Random_Forest Grid

from sklearn.ensemble import RandomForestClassifier

p_grid = {
    "n_estimators": [80,100,120,140],
    "criterion": ["entropy"],
    "min_samples_split": [2,20,100],
    "min_samples_leaf": [1,5,20],
    "min_weight_fraction_leaf": [0.0,0.0001,0.001,0.01],
    "max_features": ["sqrt"],
    "min_impurity_decrease": [0.0,0.0001,0.001,0.01],
    "ccp_alpha": [0.0,0.1,1.0],
}

Forest_grid = HalvingGridSearchCV(
    estimator=RandomForestClassifier(),
    param_grid=p_grid,
    scoring='accuracy',
    max_resources=500,
    n_jobs=2)

if(Forest):
    Forest_grid.fit(data_training_X,data_training_Y)
    results = pandas.DataFrame(Forest_grid.cv_results_)

# Compact view of param, 10 worst
results[cols].sort_values("rank_test_score").tail(10)

In [ ]:
# Compact view of param, 10 best
results[cols].sort_values("rank_test_score").head(10)

In [ ]:
#Hist_Grabient_Boosting Grid

from sklearn.ensemble import HistGradientBoostingClassifier

p_grid = {
    "learning_rate":[0.03,0.04],
    "max_iter":[180],
    "max_leaf_nodes":[31],
    "min_samples_leaf":[30],
    "max_features":[0.4,0.5,0.6],
    "categorical_features":[["Urban_or_Rural_Area","Did_Police_Officer_Attend_Scene_of_Accident"]],
    "scoring":['accuracy'],
    "early_stopping":[False],
    "validation_fraction":[0.001],
}

#HalvingGridSearchCV / GridSearchCV
H_GB_grid = GridSearchCV(
    estimator=HistGradientBoostingClassifier(),
    param_grid=p_grid,
    scoring='accuracy',
    #max_resources=1000,
    n_jobs=2)

if(HistGB):
    H_GB_grid.fit(data_training_X,data_training_Y)
    results = pandas.DataFrame(H_GB_grid.cv_results_)

# Compact view of param, 10 worst
results[cols].sort_values("rank_test_score").tail(10)


In [ ]:
# Compact view of param, 10 best
results[cols].sort_values("rank_test_score").head(10)

In [ ]:
#SVM_(SVC) Grid

from sklearn.svm import SVC

p_grid = {
    "C":[1.0], 
    "kernel":['rbf'], 
    "degree":[3], 
    "gamma":['scale'], 
    "coef0":[0.0], 
    "shrinking":[True], 
    "probability":['deprecated'], 
    "max_iter":[-1], 
    "decision_function_shape":['ovr'], 
    "break_ties":[False], 
}

SVM_grid = HalvingGridSearchCV(
    estimator=SVC(),
    param_grid=p_grid,
    scoring='accuracy',
    max_resources=500,
    n_jobs=2)

if(SVM):
    SVM_grid.fit(data_training_X,data_training_Y)
    results = pandas.DataFrame(SVM_grid.cv_results_)

# Compact view of param, 10 worst
results[cols].sort_values("rank_test_score").tail(10)

In [ ]:
# Compact view of param, 10 best
results[cols].sort_values("rank_test_score").head(10)